# Procesamiento de Imágenes, audio y vídeo
## Práctica 5. Detección de características
### Ejercicio 1.  Desarrolle una aplicación que permita
**a)  A través de la interfaz modificar los parámetros del detector de características SIFT.**

**b) Seleccionar un área de interés en una imagen de elección una imagen de naturaleza médica y una imagen telemétrica.**

**c) Buscar esa área de interés (recuádrela en rojo) dentro de diferentes versiones de la imagen de partida (con cambios de traslación, escala y rotación) [NOTA: Estos cambios se pueden acometer con un editor de imágenes o con el trabajo hecho en prácticas previas]. Altere mediante la interfaz las configuraciones de parámetros para mejorar la detección.**

**d) Pruebe a hacer lo mismo que en el apartado c) con diferentes grados de deformación de la imagen. [NOTA: Estos cambios se pueden acometer con un editor de imágenes o con el trabajo hecho en prácticas previas].**

In [2]:
"""
Práctica 5 - Detección de Características con SIFT
Aplicación unificada para trabajar con múltiples imágenes y ROIs
"""

import cv2 as cv
import numpy as np
from pathlib import Path
import time
from enum import Enum

# ============================================================================
# CONFIGURACIÓN GLOBAL
# ============================================================================

class ImageType(Enum):
    """Tipos de imagen soportados"""
    LIBRE = 0
    MEDICA = 1
    TELEMETRICA = 2

class AppConfig:
    """Configuración de la aplicación"""
    # Rutas de imágenes (MODIFICAR SEGÚN TUS ARCHIVOS)
    IMAGES = {
        ImageType.LIBRE: "images/super-mario-bros-nintendo.jpg",
        ImageType.MEDICA: "images/mri/Te-gl_0030.jpg",
        ImageType.TELEMETRICA: "images/satellite/image186.bmp"
    }
    
    # Parámetros SIFT por defecto
    DEFAULT_SIFT = {
        'nfeatures': 1000,
        'nOctaveLayers': 3,
        'contrastThreshold': 0.04,
        'edgeThreshold': 10.0,
        'sigma': 1.6
    }
    
    # Carpetas de salida
    OUTPUT_DIR = Path("output")
    ROI_DIR = Path("output/rois")
    TRANSFORMED_DIR = Path("output/transformed")
    RESULTS_DIR = Path("output/results")

# ============================================================================
# CLASE PRINCIPAL - GESTOR DE APLICACIÓN
# ============================================================================

class SIFTDetectionApp:
    """Aplicación principal para detección de características SIFT"""
    
    def __init__(self):
        self.config = AppConfig()
        self.setup_directories()
        self.current_image_type = ImageType.LIBRE
        self.images = {}
        self.rois = {}
        self.sift_params = self.config.DEFAULT_SIFT.copy()
        
    def setup_directories(self):
        """Crear directorios de salida"""
        for directory in [self.config.OUTPUT_DIR, self.config.ROI_DIR, 
                         self.config.TRANSFORMED_DIR, self.config.RESULTS_DIR]:
            directory.mkdir(parents=True, exist_ok=True)
    
    def load_images(self):
        """Cargar todas las imágenes"""
        print("=" * 60)
        print("Cargando imágenes...")
        for img_type, path in self.config.IMAGES.items():
            img = cv.imread(path)
            if img is not None:
                self.images[img_type] = img
                print(f"✓ {img_type.name}: {path}")
            else:
                print(f"✗ {img_type.name}: No encontrada en {path}")
        print("=" * 60)
        return len(self.images) > 0
    
    def run(self):
        """Ejecutar menú principal"""
        if not self.load_images():
            print("ERROR: No se pudo cargar ninguna imagen")
            return
        
        while True:
            print("\n" + "=" * 60)
            print("MENÚ PRINCIPAL - DETECCIÓN DE CARACTERÍSTICAS SIFT")
            print("=" * 60)
            print("1. Ajustar parámetros SIFT (Apartado A)")
            print("2. Seleccionar ROIs (Apartado B)")
            print("3. Detectar ROI en transformaciones (Apartados C y D)")
            print("4. Cambiar imagen actual")
            print("5. Generar imágenes transformadas")
            print("0. Salir")
            print("=" * 60)
            print(f"Imagen actual: {self.current_image_type.name}")
            print(f"ROIs guardadas: {len(self.rois)}")
            
            choice = input("\nSeleccione opción: ").strip()
            
            if choice == '1':
                self.adjust_sift_parameters()
            elif choice == '2':
                self.select_rois()
            elif choice == '3':
                self.detect_roi_in_transformations()
            elif choice == '4':
                self.change_current_image()
            elif choice == '5':
                self.generate_transformations()
            elif choice == '0':
                print("Saliendo...")
                break
            else:
                print("Opción no válida")
    
    # ========================================================================
    # APARTADO A - AJUSTE DE PARÁMETROS SIFT
    # ========================================================================
    
    def adjust_sift_parameters(self):
        """Interfaz interactiva para ajustar parámetros SIFT"""
        if self.current_image_type not in self.images:
            print("No hay imagen cargada para este tipo")
            return
        
        img = self.images[self.current_image_type]
        gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
        win = "SIFT - Ajuste de Parámetros"
        
        cv.namedWindow(win, cv.WINDOW_NORMAL)
        cv.resizeWindow(win, 1280, 720)
        
        # Crear trackbars
        cv.createTrackbar("nfeatures", win, self.sift_params['nfeatures'], 10000, lambda x: None)
        cv.createTrackbar("nOctaveLayers", win, self.sift_params['nOctaveLayers'], 10, lambda x: None)
        cv.createTrackbar("contrast x1000", win, int(self.sift_params['contrastThreshold'] * 1000), 1000, lambda x: None)
        cv.createTrackbar("edgeThreshold", win, int(self.sift_params['edgeThreshold']), 100, lambda x: None)
        cv.createTrackbar("sigma x100", win, int(self.sift_params['sigma'] * 100), 500, lambda x: None)
        
        fps_history = []
        font = cv.FONT_HERSHEY_SIMPLEX
        
        print("\nControles:")
        print("  's' - Guardar configuración actual")
        print("  'r' - Restaurar valores por defecto")
        print("  'q' o ESC - Salir")
        
        while True:
            t0 = time.time()
            
            # Leer valores
            nfeatures = cv.getTrackbarPos("nfeatures", win)
            nOctaveLayers = max(1, cv.getTrackbarPos("nOctaveLayers", win))
            contrast_raw = cv.getTrackbarPos("contrast x1000", win)
            edge_raw = cv.getTrackbarPos("edgeThreshold", win)
            sigma_raw = cv.getTrackbarPos("sigma x100", win)
            
            # Convertir a valores reales
            contrastThreshold = max(0.001, contrast_raw / 1000.0)
            edgeThreshold = max(1.0, float(edge_raw))
            sigma = max(0.5, sigma_raw / 100.0)
            
            # Crear detector SIFT
            sift = cv.SIFT_create(
                nfeatures=nfeatures,
                nOctaveLayers=nOctaveLayers,
                contrastThreshold=contrastThreshold,
                edgeThreshold=edgeThreshold,
                sigma=sigma
            )
            
            # Detectar keypoints
            kps, _ = sift.detectAndCompute(gray, None)
            vis = cv.drawKeypoints(img, kps, None, 
                                   flags=cv.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
            
            # Calcular FPS
            dt = time.time() - t0
            fps = 1.0 / dt if dt > 0 else 0.0
            fps_history.append(fps)
            if len(fps_history) > 20:
                fps_history.pop(0)
            fps_avg = sum(fps_history) / len(fps_history)
            
            # Overlay de información
            info_lines = [
                f"Imagen: {self.current_image_type.name}",
                f"Keypoints detectados: {len(kps)}",
                "",
                f"nfeatures = {nfeatures}",
                f"nOctaveLayers = {nOctaveLayers}",
                f"contrastThreshold = {contrastThreshold:.4f}",
                f"edgeThreshold = {edgeThreshold:.1f}",
                f"sigma = {sigma:.2f}",
                "",
                f"FPS: {fps_avg:.1f}",
                "",
                "s: guardar | r: reset | q/ESC: salir"
            ]
            
            y = 24
            for line in info_lines:
                cv.putText(vis, line, (12, y), font, 0.6, (0, 0, 0), 3, cv.LINE_AA)
                cv.putText(vis, line, (12, y), font, 0.6, (255, 255, 255), 1, cv.LINE_AA)
                y += 26
            
            cv.imshow(win, vis)
            key = cv.waitKey(1) & 0xFF
            
            if key in (27, ord('q')):
                break
            elif key == ord('s'):
                self.sift_params = {
                    'nfeatures': nfeatures,
                    'nOctaveLayers': nOctaveLayers,
                    'contrastThreshold': contrastThreshold,
                    'edgeThreshold': edgeThreshold,
                    'sigma': sigma
                }
                print(f"\n✓ Parámetros guardados: {self.sift_params}")
            elif key == ord('r'):
                self.sift_params = self.config.DEFAULT_SIFT.copy()
                cv.setTrackbarPos("nfeatures", win, self.sift_params['nfeatures'])
                cv.setTrackbarPos("nOctaveLayers", win, self.sift_params['nOctaveLayers'])
                cv.setTrackbarPos("contrast x1000", win, int(self.sift_params['contrastThreshold'] * 1000))
                cv.setTrackbarPos("edgeThreshold", win, int(self.sift_params['edgeThreshold']))
                cv.setTrackbarPos("sigma x100", win, int(self.sift_params['sigma'] * 100))
                print("\n✓ Parámetros restaurados a valores por defecto")
        
        cv.destroyAllWindows()
    
    # ========================================================================
    # APARTADO B - SELECCIÓN DE ROIs
    # ========================================================================
    
    def select_rois(self):
        """Seleccionar ROIs en todas las imágenes cargadas"""
        print("\n" + "=" * 60)
        print("SELECCIÓN DE ÁREAS DE INTERÉS (ROIs)")
        print("=" * 60)
        print("Instrucciones:")
        print("  - Click y arrastre para seleccionar área")
        print("  - 's' para guardar ROI actual")
        print("  - 'c' para limpiar selección")
        print("  - 'n' para siguiente imagen")
        print("  - 'q' o ESC para terminar")
        print("=" * 60)
        
        for img_type, img in self.images.items():
            print(f"\nSeleccionando ROI en: {img_type.name}")
            roi = self._select_roi_interactive(img, img_type.name)
            
            if roi is not None:
                x, y, w, h = roi
                roi_img = img[y:y+h, x:x+w]
                roi_path = self.config.ROI_DIR / f"roi_{img_type.name.lower()}.png"
                cv.imwrite(str(roi_path), roi_img)
                self.rois[img_type] = {
                    'coords': roi,
                    'image': roi_img,
                    'path': roi_path
                }
                print(f"✓ ROI guardada: {roi_path}")
            else:
                print(f"✗ No se seleccionó ROI para {img_type.name}")
    
    def _select_roi_interactive(self, img, title):
        """Interfaz interactiva para seleccionar ROI"""
        clone = img.copy()
        display = img.copy()
        win = f"Selección ROI - {title}"
        
        roi_coords = {'start': None, 'end': None, 'drawing': False, 'roi': None}
        
        def mouse_callback(event, x, y, flags, param):
            if event == cv.EVENT_LBUTTONDOWN:
                roi_coords['drawing'] = True
                roi_coords['start'] = (x, y)
                roi_coords['end'] = (x, y)
            
            elif event == cv.EVENT_MOUSEMOVE and roi_coords['drawing']:
                display[:] = clone.copy()
                roi_coords['end'] = (x, y)
                cv.rectangle(display, roi_coords['start'], roi_coords['end'], (0, 0, 255), 2)
                
                # Mostrar dimensiones
                w = abs(x - roi_coords['start'][0])
                h = abs(y - roi_coords['start'][1])
                text = f"{w}x{h}px"
                cv.putText(display, text, (x + 10, y - 10), 
                          cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
            
            elif event == cv.EVENT_LBUTTONUP:
                roi_coords['drawing'] = False
                roi_coords['end'] = (x, y)
                
                x1, y1 = roi_coords['start']
                x2, y2 = roi_coords['end']
                
                x_min, x_max = min(x1, x2), max(x1, x2)
                y_min, y_max = min(y1, y2), max(y1, y2)
                
                if x_max - x_min > 10 and y_max - y_min > 10:
                    roi_coords['roi'] = (x_min, y_min, x_max - x_min, y_max - y_min)
                    cv.rectangle(display, (x_min, y_min), (x_max, y_max), (0, 255, 0), 2)
                    cv.putText(display, "ROI OK (pulsa 's' para guardar)", (10, 30),
                              cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        
        cv.namedWindow(win, cv.WINDOW_NORMAL)
        cv.resizeWindow(win, 1024, 768)
        cv.setMouseCallback(win, mouse_callback)
        
        while True:
            cv.imshow(win, display)
            key = cv.waitKey(1) & 0xFF
            
            if key in (27, ord('q'), ord('n')):
                break
            elif key == ord('s') and roi_coords['roi'] is not None:
                cv.destroyWindow(win)
                return roi_coords['roi']
            elif key == ord('c'):
                display[:] = clone.copy()
                roi_coords = {'start': None, 'end': None, 'drawing': False, 'roi': None}
        
        cv.destroyWindow(win)
        return None
    
    # ========================================================================
    # APARTADO C y D - DETECCIÓN EN TRANSFORMACIONES
    # ========================================================================
    
    def detect_roi_in_transformations(self):
        """Detectar ROI en imágenes transformadas"""
        if not self.rois:
            print("\nERROR: Primero debe seleccionar ROIs (opción 2)")
            input("Presione Enter para continuar...")
            return
        
        print("\n" + "=" * 60)
        print("DETECCIÓN DE ROI EN IMÁGENES TRANSFORMADAS")
        print("=" * 60)
        print("Seleccione el tipo de imagen:")
        for i, img_type in enumerate(ImageType):
            status = "✓" if img_type in self.rois else "✗"
            print(f"  {i + 1}. {status} {img_type.name}")
        
        choice = input("\nOpción: ").strip()
        try:
            img_type = list(ImageType)[int(choice) - 1]
        except:
            print("Opción inválida")
            input("Presione Enter para continuar...")
            return
        
        if img_type not in self.rois or img_type not in self.images:
            print(f"No hay ROI o imagen para {img_type.name}")
            input("Presione Enter para continuar...")
            return
        
        # Buscar imágenes transformadas
        pattern = f"{img_type.name.lower()}_*.jpg"
        transformed_files = list(self.config.TRANSFORMED_DIR.glob(pattern))
        
        if not transformed_files:
            print(f"\nNo se encontraron imágenes transformadas para {img_type.name}")
            print(f"Patrón de búsqueda: {self.config.TRANSFORMED_DIR / pattern}")
            print(f"Use la opción 5 para generar transformaciones")
            input("Presione Enter para continuar...")
            return
        
        print(f"\n✓ Encontradas {len(transformed_files)} imágenes transformadas")
        print("\nControles:")
        print("  ESPACIO o ENTER - Siguiente imagen")
        print("  's' - Guardar resultado")
        print("  'q' o ESC - Volver al menú")
        input("\nPresione Enter para comenzar...")
        
        roi_data = self.rois[img_type]
        roi_img = cv.cvtColor(roi_data['image'], cv.COLOR_BGR2GRAY)
        
        # Crear SIFT con parámetros actuales
        print("\nCreando detector SIFT con parámetros:")
        for key, value in self.sift_params.items():
            print(f"  {key}: {value}")
        
        sift = cv.SIFT_create(**self.sift_params)
        kp_roi, des_roi = sift.detectAndCompute(roi_img, None)
        
        if des_roi is None or len(kp_roi) == 0:
            print("ERROR: No se pudieron extraer descriptores de la ROI")
            input("Presione Enter para continuar...")
            return
        
        print(f"✓ ROI: {len(kp_roi)} keypoints detectados")
        
        # Configurar matcher
        FLANN_INDEX_KDTREE = 1
        index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
        search_params = dict(checks=50)
        flann = cv.FlannBasedMatcher(index_params, search_params)
        
        results_summary = []
        
        for idx, tf_path in enumerate(transformed_files):
            print(f"\n[{idx+1}/{len(transformed_files)}] Procesando: {tf_path.name}")
            
            img_test = cv.imread(str(tf_path))
            if img_test is None:
                print("  ✗ Error al cargar imagen")
                continue
            
            gray_test = cv.cvtColor(img_test, cv.COLOR_BGR2GRAY)
            kp_test, des_test = sift.detectAndCompute(gray_test, None)
            
            if des_test is None or len(kp_test) == 0:
                print(f"  ✗ No se detectaron características ({len(kp_test) if kp_test else 0} kps)")
                continue
            
            print(f"  ✓ Imagen test: {len(kp_test)} keypoints")
            
            # Matching
            try:
                matches = flann.knnMatch(des_roi, des_test, k=2)
            except cv.error as e:
                print(f"  ✗ Error en matching: {e}")
                continue
            
            # Filtro de Lowe
            good_matches = []
            for match_pair in matches:
                if len(match_pair) == 2:
                    m, n = match_pair
                    if m.distance < 0.7 * n.distance:
                        good_matches.append(m)
            
            print(f"  → Coincidencias buenas: {len(good_matches)}")
            
            # Visualización
            result = img_test.copy()
            MIN_MATCH_COUNT = 10
            detection_status = "NO DETECTADA"
            
            if len(good_matches) >= MIN_MATCH_COUNT:
                src_pts = np.float32([kp_roi[m.queryIdx].pt for m in good_matches]).reshape(-1, 1, 2)
                dst_pts = np.float32([kp_test[m.trainIdx].pt for m in good_matches]).reshape(-1, 1, 2)
                
                try:
                    M, mask = cv.findHomography(src_pts, dst_pts, cv.RANSAC, 5.0)
                    
                    if M is not None:
                        h, w = roi_img.shape
                        pts = np.float32([[0, 0], [0, h-1], [w-1, h-1], [w-1, 0]]).reshape(-1, 1, 2)
                        dst = cv.perspectiveTransform(pts, M)
                        
                        # Verificar que el polígono sea válido
                        if cv.contourArea(dst) > 100:  # área mínima
                            result = cv.polylines(result, [np.int32(dst)], True, (0, 0, 255), 3, cv.LINE_AA)
                            
                            cv.putText(result, f"ROI DETECTADA ({len(good_matches)} matches)", 
                                      (10, 30), cv.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
                            detection_status = "DETECTADA"
                            print(f"  ✓✓ ROI DETECTADA correctamente")
                        else:
                            cv.putText(result, "Homografia invalida (area muy pequena)", (10, 30),
                                      cv.FONT_HERSHEY_SIMPLEX, 0.7, (0, 165, 255), 2)
                    else:
                        cv.putText(result, "Homografia fallida", (10, 30),
                                  cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                except cv.error as e:
                    print(f"  ✗ Error en homografía: {e}")
                    cv.putText(result, "Error en homografia", (10, 30),
                              cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
            else:
                cv.putText(result, f"Pocas coincidencias ({len(good_matches)} < {MIN_MATCH_COUNT})", 
                          (10, 30), cv.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
            
            results_summary.append({
                'file': tf_path.name,
                'matches': len(good_matches),
                'status': detection_status
            })
            
            # Mostrar resultado
            win = f"Deteccion [{idx+1}/{len(transformed_files)}] - {tf_path.stem}"
            cv.namedWindow(win, cv.WINDOW_NORMAL)
            cv.resizeWindow(win, 1024, 768)
            cv.imshow(win, result)
            
            print(f"  Mostrando resultado... (ESPACIO=siguiente, s=guardar, q=salir)")
            
            exit_loop = False
            while True:
                key = cv.waitKey(100) & 0xFF  # timeout de 100ms para no bloquear
                
                if key == 255:  # no key pressed
                    continue
                elif key in (27, ord('q')):  # ESC o q
                    exit_loop = True
                    break
                elif key in (32, 13):  # ESPACIO o ENTER
                    break
                elif key == ord('s'):
                    out_path = self.config.RESULTS_DIR / f"detected_{tf_path.name}"
                    cv.imwrite(str(out_path), result)
                    print(f"  ✓ Guardado: {out_path}")
            
            cv.destroyWindow(win)
            
            if exit_loop:
                break
        
        # Mostrar resumen
        print("\n" + "=" * 60)
        print("RESUMEN DE DETECCIONES")
        print("=" * 60)
        for res in results_summary:
            status_icon = "✓" if res['status'] == "DETECTADA" else "✗"
            print(f"{status_icon} {res['file']}: {res['matches']} matches - {res['status']}")
        print("=" * 60)
        
        input("\nPresione Enter para volver al menú...")
        cv.destroyAllWindows()
    
    # ========================================================================
    # GENERACIÓN DE TRANSFORMACIONES
    # ========================================================================
    
    def generate_transformations(self):
        """Generar versiones transformadas de las imágenes"""
        print("\n" + "=" * 60)
        print("GENERACIÓN DE IMÁGENES TRANSFORMADAS")
        print("=" * 60)
        
        transformations = {
            'rotacion_30': lambda img: self._rotate_image(img, 30),
            'rotacion_-45': lambda img: self._rotate_image(img, -45),
            'escala_1.5': lambda img: self._scale_image(img, 1.5),
            'escala_0.7': lambda img: self._scale_image(img, 0.7),
            'traslacion_50_100': lambda img: self._translate_image(img, 50, 100),
            'traslacion_-30_80': lambda img: self._translate_image(img, -30, 80),
            'perspectiva_leve': lambda img: self._perspective_transform(img, 'leve'),
            'perspectiva_moderada': lambda img: self._perspective_transform(img, 'moderada'),
            'perspectiva_fuerte': lambda img: self._perspective_transform(img, 'fuerte'),
        }
        
        for img_type, img in self.images.items():
            print(f"\nProcesando: {img_type.name}")
            
            for tf_name, tf_func in transformations.items():
                try:
                    transformed = tf_func(img)
                    out_name = f"{img_type.name.lower()}_{tf_name}.jpg"
                    out_path = self.config.TRANSFORMED_DIR / out_name
                    cv.imwrite(str(out_path), transformed)
                    print(f"  ✓ {tf_name}")
                except Exception as e:
                    print(f"  ✗ {tf_name}: {e}")
        
        print("\n✓ Transformaciones generadas en:", self.config.TRANSFORMED_DIR)
    
    def _rotate_image(self, img, angle):
        """Rotar imagen manteniendo todo el contenido visible"""
        h, w = img.shape[:2]
        center = (w // 2, h // 2)
        
        # Calcular nueva dimensión para que quepa toda la rotación
        M = cv.getRotationMatrix2D(center, angle, 1.0)
        cos = np.abs(M[0, 0])
        sin = np.abs(M[0, 1])
        
        new_w = int((h * sin) + (w * cos))
        new_h = int((h * cos) + (w * sin))
        
        # Ajustar matriz de transformación
        M[0, 2] += (new_w / 2) - center[0]
        M[1, 2] += (new_h / 2) - center[1]
        
        return cv.warpAffine(img, M, (new_w, new_h), borderValue=(255, 255, 255))
    
    def _scale_image(self, img, scale):
        """Escalar imagen con padding para mantener tamaño original"""
        h, w = img.shape[:2]
        scaled = cv.resize(img, None, fx=scale, fy=scale, interpolation=cv.INTER_LINEAR)
        
        # Si es más grande, recortar al centro
        if scale > 1.0:
            new_h, new_w = scaled.shape[:2]
            start_y = (new_h - h) // 2
            start_x = (new_w - w) // 2
            return scaled[start_y:start_y+h, start_x:start_x+w]
        # Si es más pequeña, añadir padding
        else:
            new_h, new_w = scaled.shape[:2]
            result = np.full((h, w, 3), 255, dtype=np.uint8)
            start_y = (h - new_h) // 2
            start_x = (w - new_w) // 2
            result[start_y:start_y+new_h, start_x:start_x+new_w] = scaled
            return result
    
    def _translate_image(self, img, tx, ty):
        """Trasladar imagen"""
        h, w = img.shape[:2]
        M = np.float32([[1, 0, tx], [0, 1, ty]])
        return cv.warpAffine(img, M, (w, h), borderValue=(255, 255, 255))
    
    def _perspective_transform(self, img, intensity='leve'):
        """Aplicar transformación de perspectiva"""
        h, w = img.shape[:2]
        pts1 = np.float32([[0, 0], [w-1, 0], [0, h-1], [w-1, h-1]])
        
        # Ajustar offsets según tamaño de imagen
        base_offset = min(w, h) // 20  # offset proporcional
        
        if intensity == 'leve':
            offset = base_offset
        elif intensity == 'moderada':
            offset = base_offset * 2
        else:  # fuerte
            offset = base_offset * 3
        
        pts2 = np.float32([
            [offset, offset//2],
            [w - offset*2, offset],
            [offset//2, h - offset],
            [w - offset, h - offset//2]
        ])
        
        M = cv.getPerspectiveTransform(pts1, pts2)
        return cv.warpPerspective(img, M, (w, h), borderValue=(255, 255, 255))
    
    def change_current_image(self):
        """Cambiar imagen de trabajo actual"""
        print("\nSeleccione imagen:")
        for i, img_type in enumerate(ImageType):
            status = "✓" if img_type in self.images else "✗"
            print(f"  {i + 1}. {status} {img_type.name}")
        
        choice = input("\nOpción: ").strip()
        try:
            img_type = list(ImageType)[int(choice) - 1]
            if img_type in self.images:
                self.current_image_type = img_type
                print(f"✓ Imagen actual: {img_type.name}")
            else:
                print("✗ Imagen no disponible")
        except:
            print("Opción inválida")

# ============================================================================
# PUNTO DE ENTRADA
# ============================================================================

if __name__ == "__main__":
    print("""
    ╔════════════════════════════════════════════════════════════════╗
    ║                                                                ║
    ║     PRÁCTICA 5 - DETECCIÓN DE CARACTERÍSTICAS CON SIFT        ║
    ║                                                                ║
    ║  Procesamiento de Imágenes, Audio y Vídeo                     ║
    ║                                                                ║
    ╚════════════════════════════════════════════════════════════════╝
    """)
    
    print("\nNOTA IMPORTANTE:")
    print("Antes de ejecutar, asegúrese de:")
    print("1. Modificar las rutas en AppConfig.IMAGES")
    print("2. Tener instalado opencv-contrib-python")
    print("3. Contar con 3 imágenes: libre elección, médica y telemétrica")
    print("\n" + "=" * 60)
    
    try:
        app = SIFTDetectionApp()
        app.run()
    except KeyboardInterrupt:
        print("\n\nInterrumpido por el usuario")
    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        cv.destroyAllWindows()


    ╔════════════════════════════════════════════════════════════════╗
    ║                                                                ║
    ║     PRÁCTICA 5 - DETECCIÓN DE CARACTERÍSTICAS CON SIFT        ║
    ║                                                                ║
    ║  Procesamiento de Imágenes, Audio y Vídeo                     ║
    ║                                                                ║
    ╚════════════════════════════════════════════════════════════════╝
    

NOTA IMPORTANTE:
Antes de ejecutar, asegúrese de:
1. Modificar las rutas en AppConfig.IMAGES
2. Tener instalado opencv-contrib-python
3. Contar con 3 imágenes: libre elección, médica y telemétrica

Cargando imágenes...
✓ LIBRE: images/super-mario-bros-nintendo.jpg
✓ MEDICA: images/mri/Te-gl_0030.jpg
✓ TELEMETRICA: images/satellite/image186.bmp

MENÚ PRINCIPAL - DETECCIÓN DE CARACTERÍSTICAS SIFT
1. Ajustar parámetros SIFT (Apartado A)
2. Seleccionar ROIs (Apartado B)
3. Detectar ROI en transfor